In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
#Funciones auxiliares sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold #Split y cross Validation
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score #Metricas
from sklearn.utils import shuffle 


#Gradient Boosting
import lightgbm as lgb


# Configuramos el estilo de las visualizaciones
sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")

breed_df = pd.read_csv(os.path.join(r"C:\Users\molin\OneDrive\Desktop\Maestria_Ciencia_de_Datos\Labo_2\petfinder-adoption-prediction\breed_labels.csv"))
color_df = pd.read_csv(os.path.join(r"C:\Users\molin\OneDrive\Desktop\Maestria_Ciencia_de_Datos\Labo_2\petfinder-adoption-prediction\color_labels.csv"))
state_df = pd.read_csv(os.path.join(r"C:\Users\molin\OneDrive\Desktop\Maestria_Ciencia_de_Datos\Labo_2\petfinder-adoption-prediction\state_labels.csv"))

import os

# Definimos la ruta
ruta_archivo = r"C:\Users\molin\OneDrive\Desktop\Maestria_Ciencia_de_Datos\Labo_2\petfinder-adoption-prediction\train\train.csv"

# Intentamos leer con latin-1

# Agregamos el parámetro sep=';'
train_df = pd.read_csv(ruta_archivo, sep=',', encoding='latin-1')

In [4]:
SEED = 42 #Semilla de procesos aleatorios (para poder replicar exactamente al volver a correr un modelo)
TEST_SIZE = 0.2 #Facción para train/test= split

In [5]:
#Separo un 20% para test estratificado opr target
df_train, df_test = train_test_split(train_df,
                               test_size = TEST_SIZE,
                               random_state = SEED,
                               stratify = train_df.AdoptionSpeed)

In [6]:
import pandas as pd

# 1. Cargar el dataset original
# Asegúrate de que el archivo esté en la misma carpeta que tu notebook

def categorizar_tamano(row):
    breed = str(row['BreedName']).lower()
    pet_type = row['Type']
    
    # Gatos (Type = 2): Los clasificamos como 'Pequeño' por defecto en comparación a un perro
    if pet_type == 2:
        return 'Pequeño'
        
    # Perros (Type = 1)
    if 'mixed breed' in breed:
        return 'Mediano' # Promedio estadístico seguro para perros mestizos
        
    # A. Excepciones específicas (se evalúan antes que las reglas generales)
    excepciones_grandes = ['black russian terrier', 'airedale terrier', 'afghan hound', 'bloodhound', 'greyhound', 'irish wolfhound']
    excepciones_medianos = ['american staffordshire terrier', 'pit bull terrier', 'bull terrier', 'beagle', 'basset hound', 'whippet', 'standard schnauzer', 'cocker spaniel']
    excepciones_pequenos = ['dachshund', 'miniature schnauzer', 'toy', 'miniature']
    
    if any(exc in breed for exc in excepciones_grandes):
        return 'Grande'
    if any(exc in breed for exc in excepciones_medianos):
        return 'Mediano'
    if any(exc in breed for exc in excepciones_pequenos):
        return 'Pequeño'
        
    # B. Reglas generales por familias funcionales / palabras clave
    # La mayoría de los terriers y perros de compañía son pequeños
    keywords_pequenos = ['terrier', 'chihuahua', 'pomeranian', 'pug', 'shih tzu', 'maltese', 'pekingese', 'corgi', 'bichon', 'french bulldog', 'papillon', 'havanese', 'pinscher']
    
    # La mayoría de los pastores, cobradores, mastines y sabuesos son grandes
    keywords_grandes = ['retriever', 'shepherd', 'mastiff', 'great', 'husky', 'malamute', 'akita', 'hound', 'pointer', 'setter', 'collie', 'rottweiler', 'doberman', 'boxer', 'american bulldog', 'sheepdog', 'dane', 'bernard', 'pyrenees', 'newfoundland', 'weimaraner', 'ridgeback', 'corso', 'komondor', 'kuvasz', 'borzoi', 'dalmatian']
    
    if any(kw in breed for kw in keywords_pequenos):
        return 'Pequeño'
    if any(kw in breed for kw in keywords_grandes):
        return 'Grande'
        
    # C. Todo lo que no caiga en las reglas anteriores se asume Mediano (ej. Spaniels, Bulldogs ingleses, Spitz)
    return 'Mediano'

# 2. Aplicar la función para crear la nueva columna
breed_df['SizeCategory'] = breed_df.apply(categorizar_tamano, axis=1)

# Unimos indicando que Breed1 equivale a BreedID
df_train = df_train.merge(
    breed_df[['BreedID', 'SizeCategory']], 
    left_on='Breed1', 
    right_on='BreedID', 
    how='left'
)

df_test = df_test.merge(
    breed_df[['BreedID', 'SizeCategory']], 
    left_on='Breed1', 
    right_on='BreedID', 
    how='left'
)

# Opcional: eliminar la columna BreedID duplicada que queda tras el merge
df_train = df_train.drop(columns=['BreedID'])
df_test = df_test.drop(columns=['BreedID'])
# 3. Exportar el resultado a un nuevo archivo CSV
nuevo_archivo = 'PetFinder-BreedLabels-Sized.csv'
breed_df.to_csv(nuevo_archivo, index=False)

print(f"Archivo generado exitosamente: '{nuevo_archivo}'")
print("\nMuestra de cómo quedaron las categorías:")
display(breed_df.sample(10, random_state=42))

Archivo generado exitosamente: 'PetFinder-BreedLabels-Sized.csv'

Muestra de cómo quedaron las categorías:


,BreedID,Type,BreedName,SizeCategory
183,184,1,Pumi,Mediano
60,61,1,Chinese Crested Dog,Mediano
124,125,1,Irish Wolfhound,Grande
93,94,1,Fila Brasileiro,Mediano
63,64,1,Chocolate Labrador Retriever,Grande
9,10,1,American Staffordshire Terrier,Mediano
147,148,1,Manchester Terrier,Pequeño
158,159,1,Norfolk Terrier,Pequeño
168,169,1,Pekingese,Pequeño
33,34,1,Bloodhound,Grande


In [11]:
# 1. Crear la columna usando una condición lógica 
df_train['Is_Purebred'] = np.where( (df_train['Breed2'] == 0) | (df_train['Breed1'] == df_train['Breed2']), 1, # 1 significa Sí (Es puro) 
                                                                                                           0 # 0 significa No (Es mestizo)
)
# 1. Crear la columna usando una condición lógica 
df_test['Is_Purebred'] = np.where( (df_test['Breed2'] == 0) | (df_test['Breed1'] == df_test['Breed2']), 1, # 1 significa Sí (Es puro) 
                                                                                                           0 # 0 significa No (Es mestizo)
)

In [ ]:
import numpy as np
import pandas as pd
import torch 
from functools import partial
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
import scipy as sp

# ---------------------------------------------------------
# PASO 1: PREPARACIÓN DE DATOS PARA TABNET
# ---------------------------------------------------------

# Tu lista limpia de variables categóricas
categorical_cols = [
    'Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 
    'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'State', 'SizeCategory', 'Is_Purebred'
]

features = [c for c in df_train.columns if c not in ['PetID', 'Name', 'RescuerID', 'Description']]

# TabNet necesita que todo sea numérico y saber dónde están las categorías
cat_idxs = []
cat_dims = []

for idx, col in enumerate(features):
    if col in categorical_cols:
        # Llenar nulos si los hay
        df_train[col] = df_train[col].fillna('Desconocido').astype(str)
        df_test[col] = df_test[col].fillna('Desconocido').astype(str)
        
        # Codificar de texto a enteros (0, 1, 2, 3...)
        le = LabelEncoder()
        # Entrenamos el encoder con todo junto para no perder categorías que solo estén en test
        le.fit(list(df_train[col].values) + list(df_test[col].values)) 
        
        df_train[col] = le.transform(df_train[col].values)
        df_test[col] = le.transform(df_test[col].values)
        
        cat_idxs.append(idx)
        cat_dims.append(len(le.classes_))

# TabNet exige que los datos de entrada sean arreglos de NumPy (.values), no DataFrames de Pandas
X_train_np = df_train[features].values
X_test_np = df_test[features].values
y_train_np = df_train['AdoptionSpeed'].values.reshape(-1, 1) # Obligatorio para Regresión (array 2D)

# ---------------------------------------------------------
# PASO 2: ENTRENAMIENTO CON VALIDACIÓN CRUZADA (5-FOLD)
# ---------------------------------------------------------

N_SPLITS = 5
folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds_tabnet = np.zeros((len(X_train_np), 1))
test_preds_tabnet = np.zeros((len(X_test_np), 1))

print("Iniciando entrenamiento TabNetRegressor...")

for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train_np, y_train_np)):
    print(f"\n--- Fold {fold_ + 1} ---")
    
    X_tr, y_tr = X_train_np[trn_idx], y_train_np[trn_idx]
    X_val, y_val = X_train_np[val_idx], y_train_np[val_idx]
    
    # Configuración de la red neuronal TabNet
    clf = TabNetRegressor(
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        cat_emb_dim=2, # Dimensión del embedding matemático
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size":50, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax', # El mecanismo de atención
        seed=42
    )
    
    clf.fit(
        X_train=X_tr, y_train=y_tr,
        eval_set=[(X_val, y_val)],
        eval_name=['valid'],
        eval_metric=['rmse'], # Optimizamos para RMSE porque es Regresión
        max_epochs=200,
        patience=20, # Early stopping
        batch_size=1024, virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )
    
    # Predecir valores decimales
    oof_preds_tabnet[val_idx] = clf.predict(X_val)
    test_preds_tabnet += clf.predict(X_test_np) / N_SPLITS

# ---------------------------------------------------------
# PASO 3: EL "TRUCO" MATEMÁTICO (OPTIMIZED ROUNDER)
# ---------------------------------------------------------

# Esta clase encuentra los puntos de corte perfectos para maximizar el Kappa
class OptimizedRounder(object):
    def __init__(self):
        self.coef_ = 0

    def _kappa_loss(self, coef, X, y):
        # Transforma los decimales a enteros basándose en los coeficientes iterativos
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]:
                X_p[i] = 0
            elif pred >= coef[0] and pred < coef[1]:
                X_p[i] = 1
            elif pred >= coef[1] and pred < coef[2]:
                X_p[i] = 2
            elif pred >= coef[2] and pred < coef[3]:
                X_p[i] = 3
            else:
                X_p[i] = 4
        # Devuelve el negativo porque scipy intenta minimizar, y queremos maximizar
        return -cohen_kappa_score(y, X_p, weights='quadratic')

    def fit(self, X, y):
        # Puntos de corte iniciales (0.5, 1.5, 2.5, 3.5)
        loss_partial = partial(self._kappa_loss, X=X, y=y)
        initial_coef = [0.5, 1.5, 2.5, 3.5]
        self.coef_ = sp.optimize.minimize(loss_partial, initial_coef, method='nelder-mead')

    def predict(self, X, coef):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]:
                X_p[i] = 0
            elif pred >= coef[0] and pred < coef[1]:
                X_p[i] = 1
            elif pred >= coef[1] and pred < coef[2]:
                X_p[i] = 2
            elif pred >= coef[2] and pred < coef[3]:
                X_p[i] = 3
            else:
                X_p[i] = 4
        return X_p

# Aplicamos el optimizador sobre las predicciones decimales de validación
optR = OptimizedRounder()
optR.fit(oof_preds_tabnet, y_train.values)
coefficients = optR.coef_['x']

print(f"\nCoeficientes de corte matemáticamente óptimos: {coefficients}")

# Convertimos las predicciones continuas a las 5 clases usando esos cortes
train_predictions_classes = optR.predict(oof_preds_tabnet, coefficients)

# Calculamos el Kappa final real
final_kappa = cohen_kappa_score(y_train, train_predictions_classes, weights='quadratic')
print(f"Score Kappa Final de Validación Cruzada (TabNet Regressor): {final_kappa:.4f}")

Iniciando entrenamiento TabNetRegressor...

--- Fold 1 ---


c:\Users\molin\anaconda3\envs\ldi2\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 3.15685 | valid_rmse: 1.41006 |  0:00:01s
epoch 1  | loss: 0.38783 | valid_rmse: 1.54202 |  0:00:01s
epoch 2  | loss: 0.1276  | valid_rmse: 1.54961 |  0:00:02s
epoch 3  | loss: 0.06936 | valid_rmse: 1.48708 |  0:00:03s
epoch 4  | loss: 0.04873 | valid_rmse: 1.33477 |  0:00:04s
epoch 5  | loss: 0.03668 | valid_rmse: 1.25222 |  0:00:05s
epoch 6  | loss: 0.03455 | valid_rmse: 1.16709 |  0:00:06s
epoch 7  | loss: 0.02875 | valid_rmse: 1.03793 |  0:00:07s
epoch 8  | loss: 0.02347 | valid_rmse: 1.0058  |  0:00:08s
epoch 9  | loss: 0.02374 | valid_rmse: 0.91945 |  0:00:09s
epoch 10 | loss: 0.01963 | valid_rmse: 0.71317 |  0:00:10s
epoch 11 | loss: 0.02208 | valid_rmse: 0.6632  |  0:00:11s
epoch 12 | loss: 0.02348 | valid_rmse: 0.66942 |  0:00:12s
epoch 13 | loss: 0.02607 | valid_rmse: 0.57483 |  0:00:13s
epoch 14 | loss: 0.02595 | valid_rmse: 0.43053 |  0:00:14s
epoch 15 | loss: 0.02013 | valid_rmse: 0.41153 |  0:00:15s
epoch 16 | loss: 0.01893 | valid_rmse: 0.39299 |  0:00:1

c:\Users\molin\anaconda3\envs\ldi2\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



--- Fold 2 ---


...

epoch 0  | loss: 3.50764 | valid_rmse: 1.38656 |  0:00:01s
epoch 1  | loss: 0.5473  | valid_rmse: 1.39631 |  0:00:03s
epoch 2  | loss: 0.17487 | valid_rmse: 1.53687 |  0:00:05s
epoch 3  | loss: 0.08411 | valid_rmse: 1.50163 |  0:00:07s
epoch 4  | loss: 0.04833 | valid_rmse: 1.29371 |  0:00:10s
epoch 5  | loss: 0.04636 | valid_rmse: 1.24463 |  0:00:12s
epoch 6  | loss: 0.03392 | valid_rmse: 1.07578 |  0:00:15s
epoch 7  | loss: 0.02716 | valid_rmse: 1.00939 |  0:00:17s
epoch 8  | loss: 0.02243 | valid_rmse: 0.99656 |  0:00:19s
epoch 9  | loss: 0.02257 | valid_rmse: 0.93835 |  0:00:21s
epoch 10 | loss: 0.01908 | valid_rmse: 0.87792 |  0:00:24s
epoch 11 | loss: 0.01964 | valid_rmse: 0.67426 |  0:00:26s
epoch 12 | loss: 0.01779 | valid_rmse: 0.61188 |  0:00:28s
epoch 13 | loss: 0.01712 | valid_rmse: 0.54841 |  0:00:31s
epoch 14 | loss: 0.01496 | valid_rmse: 0.58568 |  0:00:33s
epoch 15 | loss: 0.01552 | valid_rmse: 0.43765 |  0:00:36s
epoch 16 | loss: 0.02059 | valid_rmse: 0.40508 |  0:00:3

c:\Users\molin\anaconda3\envs\ldi2\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



--- Fold 4 ---


c:\Users\molin\anaconda3\envs\ldi2\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 3.03006 | valid_rmse: 1.04587 |  0:00:01s
epoch 1  | loss: 0.4406  | valid_rmse: 1.97208 |  0:00:02s
epoch 2  | loss: 0.14058 | valid_rmse: 1.69102 |  0:00:03s
epoch 3  | loss: 0.07188 | valid_rmse: 1.55413 |  0:00:04s
epoch 4  | loss: 0.04861 | valid_rmse: 1.38907 |  0:00:05s
epoch 5  | loss: 0.0347  | valid_rmse: 1.36468 |  0:00:06s
epoch 6  | loss: 0.03014 | valid_rmse: 1.17299 |  0:00:07s
epoch 7  | loss: 0.03225 | valid_rmse: 1.04188 |  0:00:08s
epoch 8  | loss: 0.02843 | valid_rmse: 0.96158 |  0:00:09s
epoch 9  | loss: 0.02501 | valid_rmse: 0.84781 |  0:00:10s
epoch 10 | loss: 0.02182 | valid_rmse: 0.80588 |  0:00:11s
epoch 11 | loss: 0.01706 | valid_rmse: 0.70682 |  0:00:12s
epoch 12 | loss: 0.02165 | valid_rmse: 0.66898 |  0:00:13s
epoch 13 | loss: 0.01831 | valid_rmse: 0.63078 |  0:00:14s
epoch 14 | loss: 0.02606 | valid_rmse: 0.56885 |  0:00:15s
epoch 15 | loss: 0.02    | valid_rmse: 0.48598 |  0:00:16s
